In [0]:
dbutils.widgets.text("catalog", "ecommerce_catalog_dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
create table if not exists ecommerce_catalog_dev.bronze.raw_orders;

In [0]:
%sql
copy into identifier(:catalog).bronze.raw_orders
from "/Volumes/" :catalog "/bronze/raw_data_volume/ecommerce_transactions/"
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'inferSchema' = 'true'
)
COPY_OPTIONS(
    'mergeSchema' = 'true'
)


In [0]:
from pyspark.sql.functions import current_timestamp, col

In [0]:
df = spark.table(f"{catalog}.bronze.raw_orders") 
df = df.withColumn("ingestion_time", current_timestamp())
df = df.withColumn("source_file", col("_metadata.file_path"))
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.bronze.raw_orders")